In [1]:
# Imports
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## Load Cluster Node Profiles

In [5]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

In [6]:
nodes = ['aws-metal-c5n', 'aws-metal-z1d', 'gpgnode-16', 'gpgnode-22', 'tu-06', 'hu-c40', 'hu-c42']
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

In [2]:
def load_profiling_runs(node, workflows, samples):
    node_profile = node_profiles[node]
    runs = {}

    for workflow, samples in zip(workflows, samples):
        runs[workflow] = {}

        for sample in samples:
            sample_dir = f'{node}/{workflow}/{sample}/'
            input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
            trace_df = pd.read_csv(sample_dir + 'trace.csv')

            runs[workflow][sample] = {
                'input': input_df,
                'trace': trace_df
            }

    return (node_profile, runs)

In [3]:
def load_workflow_execution(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    input_df = pd.read_csv(execution_dir + 'input.csv', sep=';')
    trace_df = pd.read_csv(execution_dir + 'trace.csv')

    return {
        'input': input_df,
        'trace': trace_df
    }

## Predict from TU-06

In [9]:
TU_06 = 'tu-06'
TU_06_workflows = ['rnaseq', 'chipseq']
TU_06_samples = [['sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(TU_06_profile, TU_06_runs) = load_profiling_runs(TU_06, TU_06_workflows, TU_06_samples)

print(TU_06)
print(TU_06_profile)
print(TU_06_runs)

tu-06
{'ram': 128, 'cores': 32, 'io_read': 3500.0, 'io_write': 4717.0, 'sysbench': 601.07, 'z7b': 3014.0, 'bench_time': 0.07366666666666667}
{'rnaseq': {'sample_b': {'input':                                                   name  \
0    NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...   
1    NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...   
2    NFCORE_RNASEQ:RNASEQ:FASTQ_QC_TRIM_FILTER_SETS...   
3    NFCORE_RNASEQ:RNASEQ:FASTQ_QC_TRIM_FILTER_SETS...   
4    NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_GETCHROMSI...   
..                                                 ...   
147                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   
148                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   
149                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   
150                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   
151                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   

                                 hash  \
0    28da6992782f41ecefaa5f441691b1b7   
1    28da6992782f41ecefaa5f441691b1b7   
2    

In [7]:
HU_40 = 'hu-c40'
HU_40_workflows = ['rnaseq', 'chipseq']
HU_40_samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(HU_40_profile, HU_40_runs) = load_profiling_runs(HU_40, HU_40_workflows, HU_40_samples)

In [8]:
HU_42 = 'hu-c42'
HU_42_workflows = ['rnaseq', 'chipseq']
HU_42_samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(HU_42_profile, HU_42_runs) = load_profiling_runs(HU_42, HU_42_workflows, HU_42_samples)

In [ ]:
# identify real executions on hu-cluster
hu_rnaseq = load_workflow_execution('hu-cluster', 'rnaseq', 1)
hu_chipseq = load_workflow_execution('hu-cluster', 'chipseq', 1)

In [ ]:
# hu-cluster -- rnaseq execution
for name, group in hu_rnaseq['input'].groupby('name'):
    task_name = ":".join(name.split(':')[1:])
    no_files = str(len(group))
    filepaths = group['path']
    paths = ', '.join([filepath.split('/')[-1] for filepath in filepaths])
    total_size = group['size'].sum() / 1073741824  # B to GiB
    print(f'[{task_name}]: {no_files} files [{paths}]), {total_size:.2f} GiB')

assert len(hu_rnaseq['trace']) == len(hu_rnaseq['input']['name'].unique())

[RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_FLAGSTAT (CONTROL_REP1)]: 2 files [CONTROL_REP1.sorted.bam.bai, CONTROL_REP1.sorted.bam]), 1.58 GiB
[RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_FLAGSTAT (CONTROL_REP2)]: 2 files [CONTROL_REP2.sorted.bam.bai, CONTROL_REP2.sorted.bam]), 1.58 GiB
[RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_FLAGSTAT (CONTROL_REP3)]: 2 files [CONTROL_REP3.sorted.bam, CONTROL_REP3.sorted.bam.bai]), 1.58 GiB
[RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_FLAGSTAT (TREATMENT_mIFNa11_100000_REP1)]: 2 files [TREATMENT_mIFNa11_100000_REP1.sorted.bam, TREATMENT_mIFNa11_100000_REP1.sorted.bam.bai]), 1.65 GiB
[RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_FLAGSTAT (TREATMENT_mIFNa11_100000_REP2)]: 2 files [TREATMENT_mIFNa11_100000_REP2.sorted.bam, TREATMENT_mIFNa11_100000_REP2.sorted.bam.bai]), 1.12 GiB
[RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:

In [24]:
def combine_df(input_df, trace_df):
    rows = []

    for name, group in input_df.groupby('name'):
        no_files = str(len(group))
        filepaths = group['path']
        paths = ', '.join([filepath.split('/')[-1] for filepath in filepaths])
        total_size = group['size'].sum()
        task_record = trace_df[trace_df['name'] == name]
        retain = task_record[['task_id', 'name', 'hostname', '%cpu', 'start', 'complete', 'read_bytes', 'write_bytes', 'cpu_model']]
        add = [no_files, paths, total_size]
        print(retain)
        print(add)
        break


In [25]:
# hu-c40 -- rnaseq sample_d profiling 
for name, group in HU_40_runs['rnaseq']['sample_d']['input'].groupby('name'):
    task_name = ":".join(name.split(':')[1:])
    no_files = str(len(group))
    filepaths = group['path']
    paths = ', '.join([filepath.split('/')[-1] for filepath in filepaths])
    total_size = group['size'].sum() / 1073741824  # B to GiB
    #print(f'[{task_name}]: {no_files} files [{paths}]), {total_size:.2f} GiB')
    # 'name' can be used to join the fields? we group by those values and name, and use the name to identify appropriate fields in the trace? 
    # create a new df with only the essential values from each trace
    # function should return this combined one
    # trace fields (remove non-essential)
    # add: no_files, files, file total size in bytes

assert len(HU_40_runs['rnaseq']['sample_d']['trace']) == len(HU_40_runs['rnaseq']['sample_d']['input']['name'].unique())
combine_df(HU_40_runs['rnaseq']['sample_d']['input'], HU_40_runs['rnaseq']['sample_d']['trace'])

# Make x the sum of input_df grouped by task name
x = HU_40_runs['rnaseq']['sample_d']['input'].groupby('name')['size'].sum().values
y = HU_40_runs['rnaseq']['sample_d']['trace']['duration'].values

assert len(x) == len(y)

# what is the key data
# task name
# no_files
# file types?
# file size in GiB
# train model
# then take full size in GiB for a task, make prediction
# compare prediction to actual ? 

   task_id                                               name hostname   %cpu  \
1        2  NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_GETCHROMSI...        -  100.0   

           start       complete  read_bytes  write_bytes  \
1  1767978554544  1767978563092           0        12288   

                                    cpu_model  
1  Intel(R) Xeon(R) Silver 4314 CPU @ 2.40GHz  
['1', 'genome.fa', 3147288982]


In [ ]:
# existing code below

In [ ]:
# Make a simple svm, linear regression and random forest regression model to predict y from x
X_train, X_test, y_train, y_test = train_test_split(x.reshape(-1, 1), y, test_size=0.1, random_state=42)
models = {
    'SVR': SVR(),
    'RandomForest': RandomForestRegressor(),
    'LinearRegression': LinearRegression()
}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f'{name} MSE: {mse}')


SVR MSE: 2374234354.79043
RandomForest MSE: 48398912927.08367
LinearRegression MSE: 2485924163.51955


In [8]:
# Example of how to use the model to predict duration for a new input size
new_input_size = 500000000  # 500 MB
for name, model in models.items():
    predicted_duration = model.predict([[new_input_size]])
    print(f'{name} predicted duration for input size {new_input_size} bytes: {predicted_duration[0]} seconds')


SVR predicted duration for input size 500000000 bytes: 14411.946110872112 seconds
RandomForest predicted duration for input size 500000000 bytes: 21985.805142857145 seconds
LinearRegression predicted duration for input size 500000000 bytes: 77744.10504058322 seconds
